In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings("ignore")
import os
os.makedirs("vecv_outputs", exist_ok=True)

In [2]:
OUTPUT_DIR = "vecv_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def out(filename):
    """Return the full path inside OUTPUT_DIR for any output file."""
    return os.path.join(OUTPUT_DIR, filename)

# SECTION 1: SYSTEM PARAMETERS

In [3]:
#Base operational parameters
WORKING_DAYS_PER_YEAR = 300          # Standard automotive calendar
SHIFTS_PER_DAY = 2                   # Two-shift model
HOURS_PER_SHIFT = 8                  # Hours per shift
HOURS_PER_DAY = SHIFTS_PER_DAY * HOURS_PER_SHIFT   # 16 hrs/day
WORKING_HOURS_PER_YEAR = WORKING_DAYS_PER_YEAR * HOURS_PER_DAY  # 4,800 hrs/year
ANNUAL_DEMAND = 90_000               # Annual vehicle demand (report: 90,161)

# Takt time (computed)
TAKT_TIME_HOURS = WORKING_HOURS_PER_YEAR / ANNUAL_DEMAND  # hrs/unit
TAKT_TIME_MINS = TAKT_TIME_HOURS * 60                      # = 3.20 min/unit

# Stage definitions — all times in minutes per unit (from report Table 1.2)
# Format: (stage_name, base_cycle_time_min, std_dev_min, num_machines)
STAGES = [
    ("Procurement & Inbound Logistics",  None,  None, 1),   # External; not simulated as constraint
    ("Stamping & Precision Machining",   3.00,  0.20, 1),   # SPM-automated; below takt
    ("Robotic Welding & Cab Assembly",   3.10,  0.25, 1),   # Robotic; below takt
    ("Surface Treatment & Painting",     3.05,  0.20, 1),   # Robotic CED/PU; below takt
    ("Powertrain & Final Assembly",      3.00,  0.30, 1),   # Multi-sub-assembly
    ("End-of-Line Testing & Calibration",1.50,  0.15, 1),   # Automated rolls test
    ("Outbound Distribution",           None,  None, 1),    # External; skipped
    ("Customer Delivery & Activation",  None,  None, 1),    # External; skipped
]

# Bottleneck sub-stages
# These are within Stage 5 (Powertrain & Final Assembly)
BOTTLENECK_STAGES = [
    ("Step 14: Wheel Mount",   5.43, 0.60),   # PRIMARY BOTTLENECK
    ("Wheel Alignment",        4.44, 0.50),   # Secondary constraint
    ("Brake Bleeding",         3.96, 0.40),   # Near-takt constraint
]


# SECTION 2: CORE ANALYTICAL FUNCTIONS

In [4]:
def compute_takt_time(annual_demand, working_hours_per_year):
    """
    Compute Takt time: the heartbeat of the production line.
    Takt = Available Production Time / Customer Demand Rate

    Args:
        annual_demand (int): Units required per year
        working_hours_per_year (float): Total production hours per year

    Returns:
        float: Takt time in minutes per unit
    """
    takt_minutes = (working_hours_per_year * 60) / annual_demand
    return takt_minutes


def compute_throughput(bottleneck_cycle_time_min, num_parallel_stations=1):
    """
    Compute maximum throughput limited by the bottleneck stage.
    Throughput = (60 min/hr) / (cycle_time / num_stations)

    Args:
        bottleneck_cycle_time_min (float): Cycle time at bottleneck in minutes
        num_parallel_stations (int): Number of parallel machines at bottleneck

    Returns:
        dict: Throughput per hour, per day, per year
    """
    effective_cycle_time = bottleneck_cycle_time_min / num_parallel_stations
    throughput_per_hour = 60.0 / effective_cycle_time
    throughput_per_day = throughput_per_hour * HOURS_PER_DAY
    throughput_per_year = throughput_per_hour * WORKING_HOURS_PER_YEAR
    return {
        "per_hour": throughput_per_hour,
        "per_day": throughput_per_day,
        "per_year": throughput_per_year,
        "effective_cycle_time_min": effective_cycle_time
    }


def compute_utilization(demand_rate_per_hour, capacity_per_hour):
    """
    Compute resource utilization ratio.
    Utilization (rho) = Demand Rate (lambda) / Capacity (mu)

    A utilization > 1.0 means the system cannot keep up with demand.

    Args:
        demand_rate_per_hour (float): Vehicles demanded per hour
        capacity_per_hour (float): Maximum vehicles that can be processed per hour

    Returns:
        float: Utilization ratio (0–1 = stable, >1 = overloaded)
    """
    return demand_rate_per_hour / capacity_per_hour


def compute_waiting_time_mm1(utilization, service_rate_per_hour):
    """
    Compute average waiting time using M/M/1 queueing model.
    Represents stochastic delays at high-utilization manual stations.

    W_q = rho / (mu * (1 - rho))   [hours]

    The M/M/1 model assumes Poisson arrivals and exponential service times,
    which is a reasonable approximation for manual heavy-lifting tasks
    with unpredictable crane wait times (from report section 2.3).

    Args:
        utilization (float): Utilization ratio rho (must be < 1 for stable queue)
        service_rate_per_hour (float): mu — max vehicles processed/hour

    Returns:
        float: Average waiting time in minutes (or np.inf if overloaded)
    """
    if utilization >= 1.0:
        return np.inf  # Unstable queue — grows without bound
    waiting_time_hours = utilization / (service_rate_per_hour * (1 - utilization))
    return waiting_time_hours * 60  # Convert to minutes


def compute_wip_littles_law(throughput_per_hour, avg_cycle_time_hours):
    """
    Compute Work-in-Process (WIP) inventory using Little's Law.
    L = lambda * W
    Where:
        L = average number of units in the system (WIP)
        lambda = effective throughput rate (arrival/departure rate)
        W = average time a unit spends in the system

    From the report: WIP = 18.49 vehicles/hr × 24 hrs = ~444 vehicles

    Args:
        throughput_per_hour (float): lambda — effective throughput
        avg_cycle_time_hours (float): W — total time in system

    Returns:
        float: Average WIP count (number of vehicles in process)
    """
    return throughput_per_hour * avg_cycle_time_hours


def identify_bottleneck(stage_cycle_times, num_stations_per_stage):
    """
    Identify which stage is the bottleneck by comparing effective cycle times.
    The bottleneck is the stage with the highest effective cycle time.

    Args:
        stage_cycle_times (list of float): Cycle time in minutes for each stage
        num_stations_per_stage (list of int): Parallel stations for each stage

    Returns:
        dict: bottleneck index, name, effective cycle time, and all effective CTs
    """
    effective_cts = [ct / ns for ct, ns in zip(stage_cycle_times, num_stations_per_stage)]
    bottleneck_idx = np.argmax(effective_cts)
    return {
        "index": bottleneck_idx,
        "effective_cycle_times": effective_cts,
        "bottleneck_effective_ct": effective_cts[bottleneck_idx]
    }



# SECTION 3: MONTE CARLO SIMULATION


In [5]:
def run_simulation(
    demand_rate_multiplier=1.0,
    processing_time_multiplier=1.0,
    num_machines_wheel_mount=2,
    num_machines_welding=1,
    num_machines_painting=1,
    inventory_buffer_size=10,
    n_simulations=50000,
    random_seed=42
):
    """
    Run a Monte Carlo simulation of the VECV assembly system.

    Stochastic variation is introduced at each manual/semi-automated stage
    using normal distributions calibrated to empirical data from the report.
    This captures:
      - Crane wait time variance at wheel mount (σ = 0.60 min)
      - Alignment precision dependency variance (σ = 0.50 min)
      - Operator technique variation at brake bleeding (σ = 0.40 min)

    The buffer protects the bottleneck from upstream stoppages
    (e.g., 15-min AGV sensor fault documented in report section 3.8).

    Args:
        demand_rate_multiplier (float): Scale factor on annual demand (1.0 = 90,000)
        processing_time_multiplier (float): Scale factor on all cycle times
        num_machines_wheel_mount (int): Parallel stations at Step 14
        num_machines_welding (int): Parallel robotic welding stations
        num_machines_painting (int): Parallel painting stations
        inventory_buffer_size (int): WIP buffer upstream of bottleneck
        n_simulations (int): Number of Monte Carlo samples
        random_seed (int): Reproducibility seed

    Returns:
        dict: Summary statistics for all KPIs
    """
    rng = np.random.default_rng(random_seed)

    # Derived demand parameters
    annual_demand_actual = ANNUAL_DEMAND * demand_rate_multiplier
    takt_time = compute_takt_time(annual_demand_actual, WORKING_HOURS_PER_YEAR)

    # Stage configuration
    # Format: (base_ct_min, std_dev_min, num_stations)
    stage_configs = [
        (3.00 * processing_time_multiplier, 0.20, num_machines_welding),   # Welding
        (3.10 * processing_time_multiplier, 0.25, num_machines_welding),   # Cab Assembly
        (3.05 * processing_time_multiplier, 0.20, num_machines_painting),  # Painting
        (5.43 * processing_time_multiplier, 0.60, num_machines_wheel_mount), # Wheel Mount ← bottleneck
        (4.44 * processing_time_multiplier, 0.50, 1),                       # Wheel Alignment
        (3.96 * processing_time_multiplier, 0.40, 1),                       # Brake Bleeding
        (1.50 * processing_time_multiplier, 0.15, 1),                       # Final Test
    ]

    stage_names = [
        "Robotic Welding",
        "Cab Assembly",
        "Painting (CED/PU)",
        "Step 14: Wheel Mount",
        "Wheel Alignment",
        "Brake Bleeding",
        "End-of-Line Testing"
    ]

    # Storage for simulation results
    throughputs = []
    utilizations = []
    waiting_times = []
    wip_counts = []
    bottleneck_indices = []
    hourly_outputs = []

    for sim in range(n_simulations):
        # Sample stochastic cycle times for this simulation run
        sampled_cts = []
        for base_ct, std_dev, n_stations in stage_configs:
            sampled_ct = max(0.5, rng.normal(base_ct, std_dev))  # Floor at 0.5 min
            sampled_cts.append(sampled_ct)

        n_stations_list = [cfg[2] for cfg in stage_configs]

        # Effective cycle time per station accounts for parallelism
        effective_cts = [ct / ns for ct, ns in zip(sampled_cts, n_stations_list)]

        # Identify bottleneck in this simulation run
        bn_idx = int(np.argmax(effective_cts))
        bottleneck_effective_ct = effective_cts[bn_idx]

        # Throughput is governed by bottleneck
        throughput_ph = 60.0 / bottleneck_effective_ct

        # Buffer effect: a buffer of size B can absorb ~B * bottleneck_ct minutes
        # of upstream disruption without halting the bottleneck.
        # Simulated as: buffer reduces effective downtime by a fraction.
        buffer_protection_factor = min(0.15, inventory_buffer_size * 0.01)
        effective_throughput_ph = throughput_ph * (1 + buffer_protection_factor)

        # Demand rate (vehicles/hr required)
        demand_ph = annual_demand_actual / WORKING_HOURS_PER_YEAR

        # Utilization at bottleneck
        bottleneck_capacity = 60.0 / bottleneck_effective_ct
        util = compute_utilization(demand_ph, bottleneck_capacity)
        util = min(util, 2.0)  # Cap at 200% for extreme scenarios

        # Waiting time (M/M/1 approximation for manual stage variance)
        if util < 1.0:
            wait_time = compute_waiting_time_mm1(util, bottleneck_capacity)
        else:
            wait_time = 999.0  # Effectively infinite queue

        # WIP via Little's Law (24 hr production cycle per report)
        wip = compute_wip_littles_law(effective_throughput_ph, avg_cycle_time_hours=24.0)

        throughputs.append(effective_throughput_ph)
        utilizations.append(util)
        waiting_times.append(wait_time)
        wip_counts.append(wip)
        bottleneck_indices.append(bn_idx)
        hourly_outputs.append(min(effective_throughput_ph, demand_ph))

    # Aggregate results
    results = {
        "takt_time_min": takt_time,
        "annual_demand": annual_demand_actual,
        "throughput_mean": np.mean(throughputs),
        "throughput_std": np.std(throughputs),
        "throughput_p5": np.percentile(throughputs, 5),
        "throughput_p95": np.percentile(throughputs, 95),
        "throughput_per_year_mean": np.mean(throughputs) * WORKING_HOURS_PER_YEAR,
        "utilization_mean": np.mean(utilizations),
        "utilization_std": np.std(utilizations),
        "waiting_time_mean": np.mean([w for w in waiting_times if w < 900]),
        "wip_mean": np.mean(wip_counts),
        "wip_std": np.std(wip_counts),
        "bottleneck_stage": stage_names[int(np.median(bottleneck_indices))],
        "bottleneck_index": int(np.median(bottleneck_indices)),
        "stage_names": stage_names,
        "throughput_distribution": throughputs,
        "utilization_distribution": utilizations,
        "wip_distribution": wip_counts,
        "demand_ph": annual_demand_actual / WORKING_HOURS_PER_YEAR,
    }
    return results


# SECTION 4: SENSITIVITY ANALYSIS EXPERIMENTS


In [6]:
def sensitivity_demand_rate():
    """
    Experiment 1: How does bottleneck shift and throughput change
    as annual demand increases from 50% to 200% of baseline?

    At current demand, wheel mount (5.43 min)
    is the bottleneck. As demand increases, utilization approaches 1.0
    and waiting time explodes exponentially (M/M/1 dynamics).
    """
    print("\n" + "="*70)
    print("EXPERIMENT 1: Sensitivity to Demand Rate")
    print("="*70)
    print(f"{'Demand':>10} {'Annual':>10} {'Takt(min)':>10} {'Throughput/hr':>14} "
          f"{'Utilization':>12} {'WIP':>8} {'Wait(min)':>10}")
    print("-"*80)

    multipliers = np.arange(0.5, 2.1, 0.1)
    results_list = []
    for m in multipliers:
        r = run_simulation(demand_rate_multiplier=m, n_simulations=200)
        results_list.append({
            "multiplier": m,
            "annual_demand": r["annual_demand"],
            "takt": r["takt_time_min"],
            "throughput_ph": r["throughput_mean"],
            "utilization": r["utilization_mean"],
            "wip": r["wip_mean"],
            "wait_min": r["waiting_time_mean"],
            "bottleneck": r["bottleneck_stage"]
        })
        print(f"{m:>10.1f} {r['annual_demand']:>10,.0f} {r['takt_time_min']:>10.2f} "
              f"{r['throughput_mean']:>14.2f} {r['utilization_mean']:>12.3f} "
              f"{r['wip_mean']:>8.0f} {r['waiting_time_mean']:>10.2f}")

    return pd.DataFrame(results_list)


def sensitivity_processing_time():
    """
    Experiment 2: How does throughput change as processing time improves?

    A 20% cycle time reduction at wheel mount
    (5.43 → 4.30 min) is achievable via process redesign (dedicated jib
    crane + pre-staging). This experiment sweeps that range.
    """
    print("\n" + "="*70)
    print("EXPERIMENT 2: Sensitivity to Processing Time Improvement")
    print("="*70)
    print(f"{'PT Factor':>10} {'CT (min)':>10} {'Throughput/hr':>14} "
          f"{'Annual Capacity':>16} {'vs Demand':>12}")
    print("-"*70)

    multipliers = np.arange(0.60, 1.25, 0.05)
    results_list = []
    base_ct = 5.43  # Wheel mount baseline
    for m in multipliers:
        r = run_simulation(processing_time_multiplier=m, n_simulations=200)
        effective_ct = base_ct * m
        print(f"{m:>10.2f} {effective_ct:>10.2f} {r['throughput_mean']:>14.2f} "
              f"{r['throughput_per_year_mean']:>16,.0f} "
              f"{'✓ Meets' if r['throughput_per_year_mean'] >= ANNUAL_DEMAND else '✗ Below':>12}")
        results_list.append({
            "multiplier": m,
            "effective_ct_min": effective_ct,
            "throughput_ph": r["throughput_mean"],
            "annual_capacity": r["throughput_per_year_mean"],
            "meets_demand": r["throughput_per_year_mean"] >= ANNUAL_DEMAND
        })

    return pd.DataFrame(results_list)


def sensitivity_capacity():
    """
    Experiment 3: How does adding or removing capacity (parallel machines)
    at the bottleneck affect system performance?

    Parallelizing the 4.30-min task across 2 stations
    gives effective CT of 2.15 min — below the 3.20-min takt time.
    """
    print("\n" + "="*70)
    print("EXPERIMENT 3: Sensitivity to Bottleneck Capacity (Parallel Stations)")
    print("="*70)
    print(f"{'Stations':>10} {'Eff CT(min)':>12} {'Throughput/hr':>14} "
          f"{'Annual Cap':>12} {'Utilization':>12} {'Bottleneck':>25}")
    print("-"*90)

    results_list = []
    for n in range(1, 6):
        r = run_simulation(num_machines_wheel_mount=n, n_simulations=200)
        eff_ct = 5.43 / n
        print(f"{n:>10} {eff_ct:>12.2f} {r['throughput_mean']:>14.2f} "
              f"{r['throughput_per_year_mean']:>12,.0f} "
              f"{r['utilization_mean']:>12.3f} {r['bottleneck_stage']:>25}")
        results_list.append({
            "stations": n,
            "eff_ct_min": eff_ct,
            "throughput_ph": r["throughput_mean"],
            "annual_capacity": r["throughput_per_year_mean"],
            "utilization": r["utilization_mean"],
            "bottleneck": r["bottleneck_stage"]
        })

    return pd.DataFrame(results_list)


def sensitivity_inventory_buffer():
    """
    Experiment 4: How does inventory buffer size upstream of the bottleneck
    affect throughput and WIP?

    A buffer of ~5 engines protects against a
    15-minute upstream AGV sensor fault without halting the bottleneck.
    """
    print("\n" + "="*70)
    print("EXPERIMENT 4: Sensitivity to Inventory Buffer Size")
    print("="*70)
    print(f"{'Buffer Size':>12} {'Throughput/hr':>14} {'Annual Cap':>12} {'WIP':>8}")
    print("-"*55)

    results_list = []
    for buf in [0, 2, 5, 10, 15, 20, 30, 50]:
        r = run_simulation(inventory_buffer_size=buf, n_simulations=200)
        print(f"{buf:>12} {r['throughput_mean']:>14.2f} "
              f"{r['throughput_per_year_mean']:>12,.0f} {r['wip_mean']:>8.0f}")
        results_list.append({
            "buffer_size": buf,
            "throughput_ph": r["throughput_mean"],
            "annual_capacity": r["throughput_per_year_mean"],
            "wip": r["wip_mean"]
        })

    return pd.DataFrame(results_list)


In [7]:
# ── PATCH:hardcoded save paths ──────────────────────────────────────────
import os, types, matplotlib.pyplot as plt, matplotlib.patches as mpatches

OUTPUT_DIR = "vecv_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def out(filename):
    return os.path.join(OUTPUT_DIR, filename)

def plot_bottleneck_cycle_times():
    fig, ax = plt.subplots(figsize=(12, 6))
    stage_names = ["Stamping &\nMachining","Robotic\nWelding","Painting\n(CED/PU)",
                   "Powertrain\nAssembly","Step14:\nWheel Mount","Wheel\nAlignment",
                   "Brake\nBleeding","End-of-Line\nTesting"]
    cycle_times = [3.00, 3.10, 3.05, 3.00, 5.43, 4.44, 3.96, 1.50]
    colors = ["#2ca02c" if ct <= TAKT_TIME_MINS else "#d62728" for ct in cycle_times]
    bars = ax.barh(stage_names, cycle_times, color=colors, edgecolor='white', height=0.6, alpha=0.85)
    for bar, ct in zip(bars, cycle_times):
        ax.text(ct + 0.05, bar.get_y() + bar.get_height()/2, f"{ct:.2f} min", va='center', fontsize=9)
    ax.axvline(x=TAKT_TIME_MINS, color='#ff7f0e', lw=2.5, ls='--', label=f"Takt Time = {TAKT_TIME_MINS:.2f} min/unit")
    ax.set_xlabel("Cycle Time (minutes per vehicle)", fontsize=11)
    ax.set_title("VECV Assembly Stages: Cycle Time vs Takt Time\nRed bars = bottleneck candidates (exceed Takt Time of 3.20 min)", fontsize=12, fontweight='bold')
    ax.legend(handles=[mpatches.Patch(color='#2ca02c', label='Below Takt ✓'),
                       mpatches.Patch(color='#d62728', label='Exceeds Takt ✗ (Bottleneck)'),
                       mpatches.Patch(color='#ff7f0e', label=f'Takt Time ({TAKT_TIME_MINS:.2f} min)')],
              fontsize=9, loc='lower right')
    ax.grid(True, alpha=0.3, axis='x')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(out("vecv_bottleneck_identification.png"), dpi=150, bbox_inches='tight')
    print("✓ Saved:", out("vecv_bottleneck_identification.png"))
    plt.close()

def plot_comprehensive_analysis(df_demand, df_proc, df_cap, df_buf):
    # (keep existing body — only the savefig line changes)
    import numpy as np
    fig = plt.figure(figsize=(16, 12))
    from matplotlib.gridspec import GridSpec
    fig.suptitle("VECV Supply Chain Simulation — Sensitivity Analysis Dashboard\nPithampur Assembly Plant (Bottleneck: Step 14 – Wheel Mount, 5.43 min/unit)", fontsize=14, fontweight='bold', y=0.98)
    gs = GridSpec(2, 2, figure=fig, hspace=0.40, wspace=0.35)
    C_BLU,C_ORG,C_RED,C_GRN,C_GRY = "#1f77b4","#ff7f0e","#d62728","#2ca02c","#7f7f7f"
    ax1 = fig.add_subplot(gs[0,0]); ax1b = ax1.twinx()
    ax1.plot(df_demand["annual_demand"]/1000, df_demand["throughput_ph"], color=C_BLU, lw=2, marker='o', ms=4)
    ax1b.plot(df_demand["annual_demand"]/1000, df_demand["utilization"], color=C_RED, lw=2, ls='--', marker='s', ms=4)
    ax1.axhline(y=ANNUAL_DEMAND/WORKING_HOURS_PER_YEAR, color=C_GRY, ls=':', lw=1.5)
    ax1b.axhline(y=1.0, color=C_RED, ls=':', lw=1.2, alpha=0.5)
    ax1.set_xlabel("Annual Demand (thousands)", fontsize=10); ax1.set_ylabel("Throughput (vehicles/hour)", color=C_BLU, fontsize=10)
    ax1b.set_ylabel("Utilization Ratio", color=C_RED, fontsize=10); ax1.set_title("Exp 1: Impact of Demand Rate", fontweight='bold')
    ax1.legend(handles=[mpatches.Patch(color=C_BLU,label="Throughput"),mpatches.Patch(color=C_RED,label="Utilization"),mpatches.Patch(color=C_GRY,label="Demand Rate")], fontsize=8, loc="upper left")
    ax1.grid(True, alpha=0.3)
    ax2 = fig.add_subplot(gs[0,1])
    colors_pt = [C_GRN if m else C_RED for m in df_proc["meets_demand"]]
    ax2.bar(df_proc["effective_ct_min"], df_proc["annual_capacity"]/1000, width=0.15, color=colors_pt, edgecolor='white', alpha=0.85)
    ax2.axhline(y=ANNUAL_DEMAND/1000, color=C_ORG, ls='--', lw=2); ax2.axvline(x=TAKT_TIME_MINS, color=C_BLU, ls=':', lw=1.5)
    ax2.set_xlabel("Wheel Mount Cycle Time (min/unit)", fontsize=10); ax2.set_ylabel("Annual Capacity (thousands)", fontsize=10)
    ax2.set_title("Exp 2: Processing Time Improvement", fontweight='bold')
    ax2.legend(handles=[mpatches.Patch(color=C_GRN,label="Meets Demand"),mpatches.Patch(color=C_RED,label="Below Demand"),mpatches.Patch(color=C_ORG,label="Demand Target")], fontsize=8)
    ax2.grid(True, alpha=0.3, axis='y')
    ax3 = fig.add_subplot(gs[1,0]); ax3b = ax3.twinx()
    x = df_cap["stations"]
    ax3.bar(x-0.2, df_cap["annual_capacity"]/1000, width=0.35, color=C_BLU, alpha=0.75, edgecolor='white')
    ax3b.plot(x, df_cap["utilization"], color=C_RED, lw=2, marker='^', ms=6)
    ax3.axhline(y=ANNUAL_DEMAND/1000, color=C_ORG, ls='--', lw=2, alpha=0.8); ax3b.axhline(y=1.0, color=C_RED, ls=':', lw=1, alpha=0.5)
    ax3.set_xlabel("Number of Parallel Wheel Mount Stations", fontsize=10); ax3.set_ylabel("Annual Capacity (thousands)", color=C_BLU, fontsize=10)
    ax3b.set_ylabel("Utilization at Bottleneck", color=C_RED, fontsize=10); ax3.set_title("Exp 3: Effect of Adding/Removing Capacity", fontweight='bold')
    ax3.set_xticks(x); ax3.grid(True, alpha=0.3, axis='y')
    ax3.legend(handles=[mpatches.Patch(color=C_BLU,label="Annual Capacity"),mpatches.Patch(color=C_RED,label="Utilization"),mpatches.Patch(color=C_ORG,label="Demand (90k)")], fontsize=8)
    ax4 = fig.add_subplot(gs[1,1]); ax4b = ax4.twinx()
    ax4.plot(df_buf["buffer_size"], df_buf["throughput_ph"], color=C_GRN, lw=2, marker='D', ms=5)
    ax4b.plot(df_buf["buffer_size"], df_buf["wip"], color=C_ORG, lw=2, ls='--', marker='s', ms=5)
    ax4.set_xlabel("Buffer Size (vehicles upstream of bottleneck)", fontsize=10); ax4.set_ylabel("Throughput (vehicles/hour)", color=C_GRN, fontsize=10)
    ax4b.set_ylabel("WIP Inventory (vehicles)", color=C_ORG, fontsize=10); ax4.set_title("Exp 4: Inventory Buffer Effect", fontweight='bold')
    ax4.legend(handles=[mpatches.Patch(color=C_GRN,label="Throughput"),mpatches.Patch(color=C_ORG,label="WIP Count")], fontsize=8)
    ax4.grid(True, alpha=0.3)
    plt.savefig(out("vecv_sensitivity_analysis.png"), dpi=150, bbox_inches='tight')
    print("✓ Saved:", out("vecv_sensitivity_analysis.png"))
    plt.close()

def plot_improvement_scenarios():
    scenarios = {
        "Baseline\n(1 Station, 5.43 min)": run_simulation(n_simulations=300),
        "Process Redesign\n(1 Station, 4.30 min)": run_simulation(processing_time_multiplier=0.792, n_simulations=300),
        "Task Parallelization\n(2 Stations, 5.43 min)": run_simulation(num_machines_wheel_mount=2, n_simulations=300),
        "Redesign +\nParallelization": run_simulation(processing_time_multiplier=0.792, num_machines_wheel_mount=2, n_simulations=300),
        "Full Optimization\n(Redesign+Parallel+Buffer)": run_simulation(processing_time_multiplier=0.792, num_machines_wheel_mount=2, inventory_buffer_size=20, n_simulations=300),
    }
    fig, axes = plt.subplots(1, 3, figsize=(15, 6))
    fig.suptitle("VECV Improvement Strategy Comparison (Part 3 of Report)", fontsize=13, fontweight='bold')
    names = list(scenarios.keys()); colors = ["#d62728","#ff7f0e","#1f77b4","#9467bd","#2ca02c"]
    throughputs = [s["throughput_mean"] for s in scenarios.values()]
    axes[0].barh(names, throughputs, color=colors, alpha=0.85, edgecolor='white')
    axes[0].axvline(x=ANNUAL_DEMAND/WORKING_HOURS_PER_YEAR, color='black', ls='--', lw=1.5)
    axes[0].set_xlabel("Vehicles per Hour"); axes[0].set_title("Throughput Rate"); axes[0].grid(True, alpha=0.3, axis='x')
    for i,v in enumerate(throughputs): axes[0].text(v+0.1, i, f"{v:.1f}", va='center', fontsize=9)
    utils = [s["utilization_mean"] for s in scenarios.values()]
    axes[1].barh(names, utils, color=colors, alpha=0.85, edgecolor='white')
    axes[1].axvline(x=1.0, color='red', ls='--', lw=1.5); axes[1].axvline(x=0.85, color='orange', ls=':', lw=1.5)
    axes[1].set_xlabel("Utilization Ratio (ρ)"); axes[1].set_title("Bottleneck Utilization"); axes[1].grid(True, alpha=0.3, axis='x')
    for i,v in enumerate(utils): axes[1].text(v+0.005, i, f"{v:.2f}", va='center', fontsize=9)
    wips = [s["wip_mean"] for s in scenarios.values()]
    axes[2].barh(names, wips, color=colors, alpha=0.85, edgecolor='white')
    axes[2].set_xlabel("WIP Vehicle Count (Little's Law)"); axes[2].set_title("Work-in-Process Inventory"); axes[2].grid(True, alpha=0.3, axis='x')
    for i,v in enumerate(wips): axes[2].text(v+2, i, f"{v:.0f}", va='center', fontsize=9)
    for ax in axes: ax.invert_yaxis(); ax.tick_params(labelsize=8)
    plt.tight_layout()
    plt.savefig(out("vecv_improvement_scenarios.png"), dpi=150, bbox_inches='tight')
    print("✓ Saved:", out("vecv_improvement_scenarios.png"))
    plt.close()

print("✓ All plot functions patched ")

✓ All plot functions patched 


# SECTION 6: SUMMARY REPORT PRINTOUT

In [8]:
def print_baseline_report():
    """Print the baseline system metrics from the VECV report."""
    print("\n" + "="*70)
    print("  VECV PITHAMPUR PLANT — BASELINE SYSTEM METRICS (from Report)")
    print("="*70)

    demand_ph = ANNUAL_DEMAND / WORKING_HOURS_PER_YEAR
    bn_throughput = compute_throughput(5.43, num_parallel_stations=1)
    bn_throughput_2 = compute_throughput(5.43, num_parallel_stations=2)
    util_1 = compute_utilization(demand_ph, bn_throughput["per_hour"])
    util_2 = compute_utilization(demand_ph, bn_throughput_2["per_hour"])
    wip = compute_wip_littles_law(18.49, 24.0)

    print(f"\n  Annual Demand (FY25):         {ANNUAL_DEMAND:>10,} vehicles")
    print(f"  Working Hours/Year:           {WORKING_HOURS_PER_YEAR:>10,} hours")
    print(f"  Takt Time:                    {TAKT_TIME_MINS:>10.2f} min/unit")
    print(f"  Required Throughput:          {demand_ph:>10.2f} vehicles/hour")
    print()
    print(f"  --- Bottleneck: Step 14 (Wheel Mount) ---")
    print(f"  Cycle Time:                   {5.43:>10.2f} min/unit")
    print(f"  Max Throughput (1 station):   {bn_throughput['per_hour']:>10.2f} vehicles/hour")
    print(f"  Annual Capacity (1 station):  {bn_throughput['per_year']:>10,.0f} vehicles")
    print(f"  Annual Capacity (2 stations): {bn_throughput_2['per_year']:>10,.0f} vehicles")
    print(f"  Utilization (1 station):      {util_1:>10.3f}  {'⚠ OVERLOADED' if util_1 > 1 else '✓'}")
    print(f"  Utilization (2 stations):     {util_2:>10.3f}  {'⚠ OVERLOADED' if util_2 > 1 else '✓'}")
    print()
    print(f"  --- Little's Law ---")
    print(f"  Avg WIP (24hr cycle):         {wip:>10.0f} vehicles in process")
    print(f"  Inventory Holding Days:       {41:>10}  (from financial disclosures)")
    print("="*70)


# SECTION 7: MAIN EXECUTION

In [9]:
if __name__ == "__main__":
    print("\n" + "="*70)
    print("  VECV Supply Chain Simulation & Sensitivity Analysis")
    print("  Python Simulation Model — Reproducible Analysis")
    print("="*70)

    # Step 1: Baseline report
    print_baseline_report()

    # Step 2: Run all 4 sensitivity experimentsD
    df_demand = sensitivity_demand_rate()
    df_proc = sensitivity_processing_time()
    df_cap = sensitivity_capacity()
    df_buf = sensitivity_inventory_buffer()

    # Step 3: Generate all visualizations
    print("\n--- Generating Visualizations ---")
    plot_bottleneck_cycle_times()
    plot_comprehensive_analysis(df_demand, df_proc, df_cap, df_buf)
    plot_improvement_scenarios()

    # Step 4: Save CSV summaries
    df_demand.to_csv(out("sensitivity_demand.csv"), index=False)
    df_proc.to_csv(out("sensitivity_processing.csv"), index=False)
    df_cap.to_csv(out("sensitivity_capacity.csv"), index=False)
    df_buf.to_csv(out("sensitivity_buffer.csv"), index=False)
    print("✓ Saved: 4 CSV sensitivity analysis files")

    print("\n" + "="*70)
    print("  Simulation complete. All outputs saved to /outputs/")
    print("="*70)



  VECV Supply Chain Simulation & Sensitivity Analysis
  Python Simulation Model — Reproducible Analysis

  VECV PITHAMPUR PLANT — BASELINE SYSTEM METRICS (from Report)

  Annual Demand (FY25):             90,000 vehicles
  Working Hours/Year:                4,800 hours
  Takt Time:                          3.20 min/unit
  Required Throughput:               18.75 vehicles/hour

  --- Bottleneck: Step 14 (Wheel Mount) ---
  Cycle Time:                         5.43 min/unit
  Max Throughput (1 station):        11.05 vehicles/hour
  Annual Capacity (1 station):      53,039 vehicles
  Annual Capacity (2 stations):    106,077 vehicles
  Utilization (1 station):           1.697  ⚠ OVERLOADED
  Utilization (2 stations):          0.848  ✓

  --- Little's Law ---
  Avg WIP (24hr cycle):                444 vehicles in process
  Inventory Holding Days:               41  (from financial disclosures)

EXPERIMENT 1: Sensitivity to Demand Rate
    Demand     Annual  Takt(min)  Throughput/hr  Utilizat